# Ingestión del archivo `genre.csv`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo CSV usando `DataFrameReader` de Spark

In [0]:
from pyspark.sql.types import *
movie_schema = StructType([
    StructField("genreID", IntegerType(), True),
    StructField("genreName", StringType(), True)
])

genre_df = (spark.read 
    .schema(movie_schema)
    .option("header", True) 
    .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")
)

## 2. Seleccionar solo las columnas requeridas

In [0]:
from pyspark.sql.functions import col
genres_selected_df = genre_df.select(col("genreID"), col("genreName"))

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
genres_renamed_df = (genres_selected_df
    .withColumnRenamed("genreID", "genre_id")
    .withColumnRenamed("genreName", "genre_name")
)


## 4. Agregar la columna `ingestion_date` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

genres_final_df = add_ingestion_date(genres_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))

## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
genres_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
%sql
SELECT * FROM movie_silver.genres

genre_id,genre_name,ingestion_date,enviroment,file_date
12,Adventure,2026-09-13T17:50:21.856785Z,Production,2024-12-16
14,Fantasy,2026-09-13T17:50:21.856785Z,Production,2024-12-16
16,Animation,2026-09-13T17:50:21.856785Z,Production,2024-12-16
18,Drama,2026-09-13T17:50:21.856785Z,Production,2024-12-16
27,Horror,2026-09-13T17:50:21.856785Z,Production,2024-12-16
28,Action,2026-09-13T17:50:21.856785Z,Production,2024-12-16
35,Comedy,2026-09-13T17:50:21.856785Z,Production,2024-12-16
36,History,2026-09-13T17:50:21.856785Z,Production,2024-12-16
37,Western,2026-09-13T17:50:21.856785Z,Production,2024-12-16
53,Thriller,2026-09-13T17:50:21.856785Z,Production,2024-12-16
